In [1]:
from google.cloud import documentai_v1beta3 as documentai
from chromadb import Client
import chromadb
from sentence_transformers import SentenceTransformer
import json
from google.cloud import bigquery
from google.oauth2 import service_account
from google.cloud import storage
import openai
from vertexai.language_models import TextEmbeddingInput, TextEmbeddingModel
from chromadb.utils.embedding_functions import GoogleGenerativeAiEmbeddingFunction
from google import genai



/opt/anaconda3/envs/venv1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open("input_data_for_recomm.json","r") as f:
    input_data=json.load(f)
f.close()
input_data

{'billingcurrency': ['gbp'],
 'chargecategory': ['usage'],
 'chargedescription': ['n2 instance core running in americas'],
 'consumedunit': ['hours'],
 'pricingcategory': ['standard'],
 'pricingunit': ['hour'],
 'regionname': ['iowa'],
 'servicename': ['compute engine'],
 'servicecategory': ['other'],
 'cloud': ['gcp'],
 'chargeperiodstart_year': ['2025'],
 'chargeperiodstart_month': ['8'],
 'chargeperiodstart_day': ['17'],
 'billedcost': [0.9381152021838967],
 'consumedquantity': [113.62231003732063],
 'pricingquantity': [95.38947356253813],
 'listcost': [3.4156882800401713],
 'is_anomaly': [1.0]}

In [3]:
Input_Statement = f"The anomaly is detection in {input_data['cloud'][0]} cloud service. The service is {input_data['servicename'][0]}.Cost is {input_data['listcost'][0]}, with pricing category as {input_data['pricingcategory'][0]}. The region of the resource is {input_data['servicename']}. From the resource provided, frame the recommendation to prevent this cost anomaly. "

In [4]:
Input_Statement

"The anomaly is detection in gcp cloud service. The service is compute engine.Cost is 3.4156882800401713, with pricing category as standard. The region of the resource is ['compute engine']. From the resource provided, frame the recommendation to prevent this cost anomaly. "

In [5]:
with open("finops-55-0c8185e843cd.json","r") as f:
    json_data=json.load(f)
f.close()
cred = service_account.Credentials.from_service_account_info(json_data)
client = bigquery.Client(project=json_data['project_id'],credentials=cred)


In [6]:
with open("api_cred.json","r") as f:
    gcp_json=json.load(f)
f.close()
api_key_gemini = gcp_json['api_key']

In [7]:
storage_client = storage.Client(credentials=cred)

In [8]:
bucket = storage_client.bucket('pdfs-for-text-extraction')
blobs = storage_client.list_blobs('pdfs-for-text-extraction')
dir_list=[]
i=0
uri_path=[]
for blob in blobs:
    print(blob.path_helper('pdfs-for-text-extraction',blob.name))
    uri_path.append(f"gs://{blob.path_helper('pdfs-for-text-extraction',blob.name)}")
    if 'data_for_fm' in blob.name:
        blob.download_to_filename(f"data_for_fm.txt")
        break
with open("data_for_fm.txt", "r") as f:
    a= f.read()
    f.close()

pdfs-for-text-extraction/o/Compute_optimize.pdf
pdfs-for-text-extraction/o/data_for_fm.txt


In [9]:
ls=a.split('. ')
len(ls)

35

In [10]:
ls1=[]
for i in ls:
    i = i.replace("\n",' ')
    i = i.replace("\xa0",' ')
    i = i.replace("Effort •••   Savings •••",' ')
    ls1.append(i)


In [11]:
client = chromadb.PersistentClient(path="chromadb")
collection = client.get_or_create_collection(name="cloudopti3",embedding_function=GoogleGenerativeAiEmbeddingFunction(api_key
                                                                                                                      =api_key_gemini))
embedder = SentenceTransformer("all-MiniLM-L6-v2")

In [12]:
documents = ls1


In [13]:
documents = ls1

embeddings = embedder.encode(documents)
ids = [f"doc_{i}" for i in range(len(documents))]
collection.add(
    documents=documents,
    ids=ids
)


In [14]:
query=f"Best Practices for {input_data['servicename'][0]} with {input_data['pricingcategory'][0]} pricing category"

In [15]:
results = collection.query(
    query_texts=[query],
    n_results=20
)

# Output the most relevant documents
print("Most relevant documents:")
for doc in results['documents']:
    print(doc)


Most relevant documents:
[' 5 best practices for Compute Engine cost optimization When customers migrate to Google Cloud Platform (GCP), their first step is often to adopt Compute Engine, which makes it easy to procure and set up virtual machines (VMs) in the cloud that provide large amounts of computing power', 'Cloud allows you the pricing advantage of ‘pay as you go’ and also provides significant savings as you use more compute with Sustained Use Discounts', ' Recently, we expanded Compute Engine rightsizing capabilities from just individual instances to managed instance groups as well', 'However, after migrating to Google Cloud, you can baseline your usage and take advantage of deeper discounts for Compute workloads', 'Purchase commitments Our customers have diverse workloads running on Google Cloud with differing availability requirements', 'You can scale based on CPU utilization, HTTP load balancing capacity, or Stackdriver Monitoring metrics', '--END--OF--DOCUMENT--', ' If this 

In [16]:
Context = ''
for i in results['documents'][0]:
    Context = Context + i +"."

In [17]:
prompt = "Context:"+"\n"+Context+"\n"+Input_Statement

In [18]:
print(prompt)

Context:
 5 best practices for Compute Engine cost optimization When customers migrate to Google Cloud Platform (GCP), their first step is often to adopt Compute Engine, which makes it easy to procure and set up virtual machines (VMs) in the cloud that provide large amounts of computing power.Cloud allows you the pricing advantage of ‘pay as you go’ and also provides significant savings as you use more compute with Sustained Use Discounts. Recently, we expanded Compute Engine rightsizing capabilities from just individual instances to managed instance groups as well.However, after migrating to Google Cloud, you can baseline your usage and take advantage of deeper discounts for Compute workloads.Purchase commitments Our customers have diverse workloads running on Google Cloud with differing availability requirements.You can scale based on CPU utilization, HTTP load balancing capacity, or Stackdriver Monitoring metrics.--END--OF--DOCUMENT--. If this sounds like you, you are probably provi

In [19]:
client = genai.Client(api_key=api_key_gemini)
response = client.models.generate_content(
    model="gemini-1.5-flash",  
    contents=prompt
)
print("Recommendation:", response.text)

Recommendation: The cost anomaly of 3.4156882800401713 in Compute Engine, categorized as 'standard' and located in the 'compute engine' region, suggests potential over-provisioning or inefficient resource utilization.  To prevent this, several recommendations from the provided text are applicable:

1. **Rightsizing and Autoscaling:**  The document highlights the importance of rightsizing instances (both individually and within managed instance groups) and using autoscaling.  Instead of provisioning for peak capacity, leverage Compute Engine's managed instance groups to automatically scale up or down based on demand (CPU utilization, HTTP load balancing, or Stackdriver Monitoring metrics). This prevents paying for resources that aren't actively used.

2. **Utilize Sustained Use Discounts:** Leverage Google Cloud's sustained use discounts.  By consistently using Compute Engine resources, you can achieve significant savings.  Analyze your usage patterns to identify opportunities to benefi

In [210]:
# bucket = storage_client.bucket('pdfs-for-text-extraction')
# blobs = storage_client.list_blobs('pdfs-for-text-extraction')
# dir_list=[]
# i=0
# uri_path=[]
# for blob in blobs:
#     print(blob.path_helper('pdfs-for-text-extraction',blob.name))
#     uri_path.append(f"gs://{blob.path_helper('pdfs-for-text-extraction',blob.name)}")

In [209]:

# # Set up Document AI client
# client = documentai.DocumentProcessorServiceClient(credentials=cred)

# # The Google Cloud project, location, and processor ID
# project_id = 'finops-55'
# location = 'us'  # Format: 'us' or 'eu'
# processor = '3ac4c6af3f1734f9'     # Document AI processor ID

# for i in uri_path:

#     # Initialize request argument(s)
#     inline_document = documentai.Document()
#     inline_document.uri = i
#     resource_name = f'projects/{project_id}/locations/{location}/processors/{processor}'
#     request = documentai.ProcessRequest(
#         inline_document=inline_document,
#         name= resource_name,
        
#     )

#     # Make the request
#     response = client.process_document(request=request)

#     # Parse and print the recognized text
#     for i, document in enumerate(response.documents):
#         print(f"Document {i + 1} text:")
#         print(document.text)